# 08 · Evaluation

Test-set evaluation:
1. Error distribution histogram.
2. Sky-coverage error map — where does the model fail?
3. Qualitative: 5 best and 5 worst predictions.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.augmentations import build_eval_transforms
from src.data.catalog import load_hyg_catalog
from src.data.dataset import SyntheticStarFieldDataset
from src.models.astrolocnet import AstroLocNet
from src.training.metrics import angular_separation_deg_torch
from src.training.trainer import Trainer

In [ ]:
CKPT = ROOT / 'checkpoints/best.pt'
model = AstroLocNet(pretrained=False)
Trainer.load_state(model, CKPT)
model.eval()

catalog = load_hyg_catalog(ROOT / 'data/catalogs/hygdata_v3.csv', mag_limit=8.0)
ds = SyntheticStarFieldDataset(catalog, n_samples=256, transform=build_eval_transforms(224), seed=999)
loader = DataLoader(ds, batch_size=32)

preds, targets = [], []
with torch.no_grad():
    for x, y in loader:
        preds.append(model(x)); targets.append(y)
preds = torch.cat(preds); targets = torch.cat(targets)
sep = angular_separation_deg_torch(preds, targets).numpy()
print('Mean:', sep.mean(), 'Median:', np.median(sep))

## Error distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sep, bins=40, color='#7c5cff', edgecolor='white')
ax.axvline(np.median(sep), color='#ffd166', linestyle='--', label=f'median {np.median(sep):.2f}°')
ax.set_xlabel('Great-circle separation (°)'); ax.set_ylabel('# test samples')
ax.set_title('Error distribution on held-out synthetic test set'); ax.legend()
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/08_error_distribution.png', dpi=150)
plt.show()

## Error by sky region

In [ ]:
fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(111, projection='mollweide')
ra = np.deg2rad(targets[:, 0].numpy() - 180.0)
dec = np.deg2rad(targets[:, 1].numpy())
sc = ax.scatter(ra, dec, c=sep, cmap='magma_r', s=18)
fig.colorbar(sc, ax=ax, label='Angular sep (°)')
ax.grid(True, alpha=0.3)
ax.set_title('Per-sample error by sky position')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/08_error_by_sky.png', dpi=150, facecolor='white')
plt.show()